In [11]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)#自动生成了带噪声的线性回归数据  y = X·w + b + 噪声

##读取数据集

In [12]:
def load_array(data_arrays, batch_size, is_train=True): #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

#模型的定义

In [13]:
from torch import nn

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.liner=nn.Sequential(
            nn.Linear(2, 1))


    def forward(self, x):
        return self.liner(x)

##初始化模型参数

In [14]:
net = Net()
net.liner[0].weight.data.normal_(0, 0.01)
net.liner[0].bias.data.fill_(0)

loss = nn.MSELoss()#定义损失函数
trainer = torch.optim.SGD(net.parameters(), lr=0.01)#定义优化器  采用随机梯度下降

#训练函数

In [15]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.608553
epoch 2, loss 0.011563
epoch 3, loss 0.000328


In [16]:
w = net.liner[0].weight.data
print('w的估计误差:', true_w - w.reshape(true_w.shape))
b = net.liner[0].bias.data
print('b的估计误差:', true_b - b)

w的估计误差: tensor([ 0.0068, -0.0079])
b的估计误差: tensor([0.0113])
